<a href="https://colab.research.google.com/github/Shiveshrane/Research_paper_implementations/blob/main/HybridGemma3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## RoPE

In [2]:
class RoPE(nn.Module):
  def __init__(self, dim, base=10000.0, context_len=4096, batch_size=32):
    super().__init__()
    self.base=base
    self.dim=dim
    self.context_len=context_len
    self.batch_size=batch_size
    assert self.dim%2==0
    theta_num=torch.arange(0, self.dim,2).float()
    theta=1.0/(self.base**(theta_num/self.dim))
    self.register_buffer('theta', theta)

    positions=torch.arange(0, self.context_len, dtype=torch.float)
    angles=positions.unsqueeze(1)*theta.unsqueeze(0)
    self.register_buffer('cos', torch.cos(angles))
    self.register_buffer('sin', torch.sin(angles))

  def forward(self, x, start_pos=0):
    b,s,h,d=x.shape
    x=x.view(b,s,h,d//2, 2)

    cos=self.cos[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)
    sin=self.sin[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)

    x_rot=torch.stack([
        x[..., 0]*cos-x[..., 1]*sin,
        x[..., 0]*sin+x[..., 1]*cos
    ], dim=-1)
    return x_rot.view(b,s,h,d)


In [3]:
x=torch.randn(size=(1,10,4,32))
rope=RoPE(32, context_len=10)
rope(x).shape

torch.Size([1, 10, 4, 32])

## RMSNorm

In [4]:
class RMSNorm(nn.Module):
  def __init__(self, dim, eps=1e-6):
    self.gemma=nn.Parameter(torch.ones(dim))
    self.eps=eps
    self.dim=dim
  def norm(self,x):
    val=x.pow(2).mean(dim=-1, keepdim=True)
    return x/torch.sqrt(val+self.eps)
  def forward(self, x):
    return self.gemma*self.norm(x)

## KVCache

In [5]:
class KVCache(nn.Module):
  def __init__(self, num_layers):
    super().__init__()
    self.cache=[(None, None)]*num_layers
  def num_items(self, key,val, layer_id):
    return sum(1 for k,v in self.cache[layer_id] if k is not None)
  def add_item(self, key,val, layer_id):
    k_cache,v_cache=self.cache[layer_id]
    if k_cache is None:
      self.cache[layer_id]=(key, val)
    else:
      new_key=torch.cat((k_cache, key), dim=1)
      new_val=torch.cat((v_cache, val), dim=1)
      self.cache[layer_id]=(new_key, new_val)

  def get_items(self, layer_num):
    if not (0<=layer_num<len(self.cache)):
      raise ValueError(f"Layer index {layer_num} is out of range for a KV cache with {len(self.cache)} layers.")
    return self.cache[layer_num]


## Swish

In [6]:
class Swish(nn.Module):
  def __init__(self, beta=1):
    self.beta=beta
    self.sigmoid=nn.Sigmoid()
  def forward(self, x):
    return x*self.sigmoid(self.beta*x)

# Deepseek MOE (My twist, use FFN for normal use)

## Single FFN

In [7]:
class FFN(nn.Module):
  def __init__(self, block_size, embed_dims, device):
    super().__init__()
    self.hidden_dim=((embed_dims*2)*4)//3
    self.linear1=nn.Linear(embed_dims, self.hidden_dim, bias=False, device=device)
    self.linear2=nn.Linear(embed_dims, self.hidden_dim, bias=False, device=device)
    self.linear3=nn.Linear(self.hidden_dim, embed_dims, bias=False, device=device)
    self.block_size=block_size
    self.swish=Swish()
  def forward(self,x):
    x1=self.linear1(x)
    x2=self.linear2(x)
    hidden=torch.mul(self.swish(x1), x2)
    output=self.linear3(hidden)
    return output

## DeepseekMOE logic

In [8]:
class DeepSeekMOE(nn.Module):
  def __init__(self, embed_dims, block_size, experts, top_experts, device, use_shared_expert=True, noisy_topk=False, use_checkpointing=False):
    super().__init__()
    self.embed_dims = embed_dims
    self.experts = experts
    self.experts_ffns = nn.ModuleList([
        FFN(block_size=block_size, embed_dims=embed_dims, device=device)
        for _ in range(self.experts)
    ])
    self.device = device
    self.use_shared_expert=use_shared_expert
    self.gate=nn.Linear(self.embed_dims, experts, bias=False, device=device)
    if use_shared_expert:
      self.shared_expert=FFN(block_size=block_size, embed_dims=embed_dims, device=device)
    else:
      self.shared_expert=None
    if noisy_topk==True and use_checkpointing==False:
      self.noisy=nn.Linear(embed_dims, experts, bias=False, device=device)
      self.noisy_router=None
    self.noisy_topk=noisy_topk
    self.use_checkpointing=use_checkpointing
    self.top_experts=top_experts

  def forward(self, x):
    batch_size, seq_len, embed_dim=x.shape
    self.gate_op=self.gate(x)
    if self.noisy_topk==True:
      noise_op=self.noisy(x)
      gaussian_noise=torch.normal(0,1, size=self.gate_op.shape, device=self.device)
      self.noisy_router=F.softplus(noise_op)*gaussian_noise
      self.gate_op+=self.noisy_router

    shared_op=0
    output=0
    top_k=self.top_experts
    top_k_vals, top_k_indices=torch.topk(self.gate_op, k=top_k, dim=-1)
    masked=torch.full_like(self.gate_op, fill_value=float('-inf'), device=self.device)
    masked_values=masked.scatter_(-1, index=top_k_indices, src=top_k_vals)
    probs=torch.nn.functional.softmax(masked_values, dim=-1)

    outputs=torch.zeros_like(x)
    if self.use_shared_expert and self.shared_expert is not None:
      shared_op+=self.shared_expert(x)
    flat_x=x.view(-1, x.size(-1))


    for i in range(self.experts):
      expert_i_is_chosen_mask=(top_k_indices==i).any(dim=-1)
      if not expert_i_is_chosen_mask.any():
        continue
      flat_expert_i_is_chosen_mask=expert_i_is_chosen_mask.view(-1)
      selected_expert_input=flat_x[flat_expert_i_is_chosen_mask]
      expert_output_for_selected=self.experts_ffns[i](selected_expert_input)
      probs_for_exp_i=probs[:,:,i]
      token_weights=probs_for_exp_i[expert_i_is_chosen_mask]
      token_weights=token_weights.unsqueeze(-1)
      weighted_expert_op=expert_output_for_selected*token_weights

      temp_contri_from_x=torch.zeros_like(x)
      temp_contri_from_x.masked_scatter_(
          expert_i_is_chosen_mask.unsqueeze(-1).expand_as(x),
          weighted_expert_op
      )
      output+=temp_contri_from_x
      output=output+shared_op
    return output



# Logit capping (For Gemma 2, for Gemma 3 use RMS_Norm)

In [9]:
class LogitCapping(nn.Module):
  def __init__(self, soft_cap):
    super().__init__()
    self.soft_cap=soft_cap
  def forward(self, logits):
    capped_logits=self.softcap*torch.tanh(logits/self.soft_cap)
    return capped_logits

# Sliding-Window Attention

In [10]:
class SlidingWindowAttention(nn.Module):
  def __init__(self, n_heads, embed_dims,base, window_size, head_dim, max_seq_len, max_batch_size, device, qk_norm=True, dropout=0.1):
    super().__init__()
    self.n_heads=n_heads
    self.embed_dims=embed_dims
    self.window_size=window_size
    self.head_dim=head_dim
    self.max_seq_len=max_seq_len
    self.max_batch_size=max_batch_size
    self.device=device

    self.Wq=nn.Linear(in_features=embed_dims, out_features=self.n_heads*self.head_dim, bias=False, device=self.device)
    self.Wk=nn.Linear(in_features=embed_dims, out_features=self.n_heads*self.head_dim, bias=False, device=self.device)
    self.Wv=nn.Linear(in_features=embed_dims, out_features=self.n_heads*self.head_dim, bias=False, device=self.device)

    self.out=nn.Linear(in_features=self.n_heads*self.head_dim, out_features=embed_dims, bias=False, device=self.device)
    self.dropout=nn.Dropout(dropout)
    self.rope=RoPE(
        dim=self.head_dim,
        base=base,
        context_len=self.max_seq_len,
        batch_size=self.max_batch_size
    )
    self.qk_norm=qk_norm
    if self.qk_norm:
      self.q_norm=RMSNorm(self.head_dim)
      self.k_norm=RMSNorm(self.head_dim)

  def forward(self, x, layer_idx, start_pos=0, mask=None, kv_cache=None, inference=False):
    batch_size, seq_len, embed_dim=x.shape
    q=self.Wq(x)
    k=self.Wk(x)
    v=self.Wv(x)

    q=q.view(batch_size, seq_len, self.n_heads, self.head_dim)
    k=k.view(batch_size, seq_len, self.n_heads, self.head_dim)
    v=v.view(batch_size, seq_len, self.n_heads, self.head_dim)

    if self.qk_norm:
      q=self.q_norm(q)
      k=self.k_norm(k)

    q=self.rope(q, start_pos)
    k=self.rope(k, start_pos)

    if inference:
      prev_k, prev_v=kv_cache.get_items(layer_idx)
      if prev_k is not None:
        prev_k=prev_k[:, -(self.window_size-1), :,:]
        prev_v=prev_v[:, -(self.window_size-1), :,:]
        k_cat=torch.cat((prev_k, k), dim=1)
        v_cat=torch.cat((prev_v, v), dim=1)
      else:
        k_cat, v_cat=k,v

      kv_cache.add_item(k_cat, v_cat, layer_idx)
      k=k_cat
      v=v_cat

      q=q.transpose(1,2)
      k=k.transpose(1,2)
      v=v.transpose(1,2)


      attn_scores=torch.matmul(q, k.transpose(2,3))/(self.head_dim**0.5)
      attn_weights=F.softmax(attn_scores, dim=-1)
      attn_weights=self.dropout(attn_weights)

    else:
      q=q.transpose(1,2)
      k=k.transpose(1,2)
      v=v.transpose(1,2)

      attn_scores=torch.matmul(q, k.transpose(2,3))/(self.head_dim**0.5)
      seq_q,seq_k=attn_scores.shape[2], attn_scores.shape[3]
      causal_mask=torch.triu(torch.ones(seq_q, seq_k, device=self.device, dtype=torch.bool), diagonal=1)
      for i in range(seq_q):
        left_bound=max(0,i-self.window_size+1)
        causal_mask[i,:left_bound]=True

      attn_scores=attn_scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
      attn_weights=F.softmax(attn_scores, dim=-1)
      attn_weights=self.dropout(attn_weights)
    attn_output=torch.matmul(attn_weights, v)
    attn_output=attn_output.transpose(1,2).contiguous()
    attn_output=attn_output.view(batch_size, seq_len, self.n_heads*self.head_dim)
    attn_output=self.out(attn_output)
    return attn_output







# GQA

## Repeat KV

In [11]:
def repeat_kv(x, n_rep):
  b,s,h,d=x.shape
  if n_rep==1:
    return x
  x=x[:,:,:,None,:].expand(b,s,h,n_rep,d).reshape(b,s,h*n_rep,d)
  return x

## Actual Attention

In [12]:
class GQA(nn.Module):
  def __init__(self, embed_dims, n_heads, head_dim, max_seq_len, max_batch_size,n_kv_heads, qk_norm=True, base=1000000, dropout=0.1, device='cpu'):
    super().__init__()
    self.q_heads=n_heads
    self.kv_heads=n_kv_heads
    self.head_dim=head_dim
    self.embed_dim=embed_dims
    self.max_seq_len=max_seq_len
    self.qk_norm=qk_norm
    self.max_batch_size=max_batch_size
    self.base=base
    self.device=device
    self.kv_rep=self.q_heads//self.kv_heads
    self.dropout=nn.Dropout(dropout)
    self.rope=RoPE(
        dim=self.head_dim,
        base=self.base,
        context_len=self.max_seq_len,
        batch_size=self.max_batch_size
    )

    self.Wq=nn.Linear(in_features=self.embed_dim, out_features=self.q_heads*self.head_dim, bias=False, device=self.device)
    self.Wk=nn.Linear(in_features=self.embed_dim, out_features=self.kv_heads*self.head_dim, bias=False, device=self.device)
    self.Wv=nn.Linear(in_features=self.embed_dim, out_features=self.kv_heads*self.head_dim, bias=False, device=self.device)

    self.out=nn.Linear(in_features=self.q_heads*self.head_dim, out_features=self.embed_dim, bias=False, device=self.device)

    if self.qk_norm:
      self.q_norm=RMSNorm(self.head_dim)
      self.k_norm=RMSNorm(self.head_dim)
    else:
      self.q_norm=None
      self.k_norm=None
  def forward(self, x, layer_idx, start_pos=0, mask=None, kv_cache=None, inference=False):
    batch_size, seq_len, embed_dim=x.shape
    q=self.Wq(x)
    k=self.Wk(x)
    v=self.Wv(x)

    q=q.view(batch_size, seq_len, self.q_heads, self.head_dim)
    k=k.view(batch_size, seq_len, self.kv_heads, self.head_dim)
    v=v.view(batch_size, seq_len, self.kv_heads, self.head_dim)

    if self.qk_norm:
      q=self.q_norm(q)
      k=self.k_norm(k)

    q=self.rope(q, start_pos)
    k=self.rope(k, start_pos)

    if inference==True:
      kv_cache.add_item(k, v, layer_idx)
      keys,values=kv_cache.get_items(layer_idx)
    else:
      keys=k
      values=v

    keys=repeat_kv(keys, self.kv_rep)
    values=repeat_kv(values, self.kv_rep)

    q=q.transpose(1,2)
    keys=keys.transpose(1,2)
    values=values.transpose(1,2)

    attn_scores=torch.matmul(q, keys.transpose(2,3))/(self.head_dim**0.5)

    seq_q=q.shape[2]
    seq_k=q.shape[2]

    if seq_q>1 or not inference:
      mask=torch.triu(torch.ones(seq_q, seq_k, dtype=torch.bool, device=x.device), diagonal=1)
      attn_weights=attn_scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
    attn_weights=F.softmax(attn_weights, dim=-1)
    attn_weights=self.dropout(attn_weights)
    attn_output=torch.matmul(attn_weights, values)
    attn_output=attn_output.transpose(1,2).contiguous()
    attn_output=attn_output.view(batch_size, seq_len, self.q_heads*self.head_dim)
    attn_output=self.out(attn_output)
    return attn_output


# The Model_Block

In [14]:
class TransformerBlock(nn.Module):
  def __init__(self, block_no, embed_dims, n_heads, head_dim, max_seq_len, window_size, max_batch_size, n_kv_heads, experts, top_experts,use_shared_expert=True, noisy_topk=False, use_checkpointing=False, qk_norm=True,local_base=10000.0, global_base=1000000.0, dropout=0.1, device='cpu'):
    super().__init__()
    self.embed_dims=embed_dims
    self.n_heads=n_heads
    self.head_dim=head_dim
    self.max_seq_len=max_seq_len
    self.batch_size=max_batch_size
    self.window_size=window_size
    self.n_kv_heads=n_kv_heads
    self.qk_norm=qk_norm
    self.local_base=local_base
    self.global_base=global_base
    self.dropout=dropout
    self.device=device
    self.block_no=block_no
    self.experts=experts
    self.top_experts=top_experts
    self.use_shared_expert=use_shared_expert
    self.noisy_topk=noisy_topk
    self.use_checkpointing=use_checkpointing


    if ((self.block_no+1)% 6)==0:
      self.attention=GQA(
          embed_dims=self.embed_dims,
          n_heads=self.n_heads,
          head_dim=self.head_dim,
          max_seq_len=self.max_seq_len,
          max_batch_size=self.batch_size,
          n_kv_heads=self.n_kv_heads,
          qk_norm=self.qk_norm,
          base=self.global_base,
          dropout=self.dropout,
          device=self.device
      )
    else:
      self.attention=SlidingWindowAttention(
          n_heads=self.n_heads,
          embed_dims=self.embed_dims,
          base=self.local_base,
          window_size=self.window_size,
          head_dim=self.head_dim,
          max_seq_len=self.max_seq_len,
          max_batch_size=self.batch_size,
          device=self.device,
          qk_norm=self.qk_norm,
          dropout=self.dropout
      )
    self.pre_norm=RMSNorm(self.embed_dims)
    self.post_norm=RMSNorm(self.embed_dims)
    self.moe=DeepSeekMOE(
        embed_dims=self.embed_dims,
        block_size=self.max_seq_len,
        experts=experts,
        top_experts=top_experts,
        device=self.device,
        use_shared_expert=use_shared_expert
    )
    self.dropout=nn.Dropout(
        dropout
    )

  def forward(self, x, layer_idx, start_pos, mask=None, kv_cache=False, inference=False):
    b,s,d=x.shape
    res=x
    x=self.pre_norm(x)
    x=self.attention(x, layer_idx, start_pos, mask, kv_cache, inference)
    x=res+x
    x=self.post_norm(self.dropout(x))
    res=x
    x=self.moe(x)
    x=res+x
    x=self.dropout(x)
    return x

In [19]:
class HybridGemma3(nn.Module):
  def __init__(self,
               num_layers,
               vocab_size,
               embed_dims,
               n_heads,
               head_dim,
               max_seq_len,
               window_size,
               max_batch_size,
               n_kv_heads,
               experts,
               top_experts,
               use_shared_expert=True,
               noisy_topk=False,
               use_checkpointing=False,
               qk_norm=True,
               local_base=10000.0,
               global_base=1000000.0,
               dropout=0.1,
               device='cpu' ):
    super().__init__()
    self.num_layers=num_layers
    self.embed_dims=embed_dims
    self.n_heads=n_heads
    self.head_dim=head_dim
    self.max_seq_len=max_seq_len
    self.batch_size=max_batch_size
    self.window_size=window_size
    self.n_kv_heads=n_kv_heads
    self.qk_norm=qk_norm
    self.local_base=local_base
    self.global_base=global_base
    self.dropout=dropout
    self.device=device
    self.experts=experts
    self.top_experts=top_experts
    self.use_shared_expert=use_shared_expert
    self.noisy_topk=noisy_topk
    self.use_checkpointing=use_checkpointing
    self.vocab_size=vocab_size

    self.embedding=nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=self.embed_dims, device=self.device)

    self.output_layer=nn.Linear(in_features=self.embed_dims, out_features=self.vocab_size, bias=False, device=self.device)
    self.layers=nn.ModuleList([
        TransformerBlock(
            block_no=i,
            embed_dims=self.embed_dims,
            n_heads=self.n_heads,
            head_dim=self.head_dim,
            max_seq_len=self.max_seq_len,
            window_size=self.window_size,
            max_batch_size=self.batch_size,
            n_kv_heads=self.n_kv_heads,
            experts=self.experts,
            top_experts=self.top_experts,
            use_shared_expert=self.use_shared_expert,
            noisy_topk=self.noisy_topk,
            use_checkpointing=self.use_checkpointing,
            qk_norm=self.qk_norm,
            local_base=self.local_base,
            global_base=self.global_base,
            dropout=self.dropout,
            device=self.device) for i in range(self.num_layers)
    ])

    self.kv_cache=KVCache(self.num_layers)
    self.norm1=RMSNorm(self.embed_dims)
    self.norm2=RMSNorm(self.embed_dims)

  def forward(self, x,start_pos, inference=False, mask=None):
    b,s=x.shape
    x=self.embedding(x)
    x=self.norm1(x)
    for i,layer in enumerate(self.layers):
      x=layer(x, i, start_pos, mask, self.kv_cache, inference)
    x=self.norm2(x)
    x=self.output_layer(x)
    return x

